# 10 — Event Study: CG Filing Dates

Short-window abnormal-return event study around CG filing dates. Independent of `06`–`09`'s
panel/portfolio work — this asks a narrower question: does the market move around the
*disclosure* of a firm's governance report, and does the size of that move relate to the
CG score being disclosed?

**Design decisions (confirmed):**
- **Events**: every quarterly CG filing (all 4 quarters, not just Q4) for the ~247 firms in
  `matched_companies.xlsx` — maximizes power, matches the spec's "CG_score is the score for
  the filing quarter."
- **Estimation window**: (−250, −20) trading days before the filing date. Market model:
  `r_it = α + β'F_t + ε_it` on FF5+MOM factors (reconstructed to daily from
  `ff5mom_factors_monthly.csv`, same method as `08_ff5_regression.ipynb`).
- **Event windows**: (−1,+1), (−3,+3), (−5,+5), (−10,+10), (0,+5) trading days.
- **Overlapping filings**: primary results include *all* events; a separate robustness table
  excludes the second of any pair of filings (same firm) within 20 trading days of each other.
- **Multiple testing**: Romano-Wolf step-down, cluster (firm) bootstrap, B=2,000 replications,
  applied across the (sub-index × window) = 30-test family.

**Data constraint found empirically (not a design choice):** `ff5mom_factors_monthly.csv`
only covers 2022-07 through 2026-04. A (−250,−20 td) estimation window is therefore only
*fully* covered by factor data for filings from **2023-07-06** onward — earlier filings are
dropped from the estimation sample, not silently included with a partial window. This caps
the usable event sample at roughly FY24–FY26 filings (~2,500 events after all availability
filters), reported explicitly in the funnel below.

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import statsmodels.api as sm
import yfinance as yf
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from scipy import stats

BASE = Path.cwd()
RAW  = BASE / 'data' / 'raw'
PROC = BASE / 'data' / 'processed'

plt.rcParams.update({'figure.dpi': 120, 'font.size': 10,
                     'axes.spines.top': False, 'axes.spines.right': False})
sns.set_theme(style='whitegrid')

RF_ANNUAL, RF_DAILY = 0.065, 0.065 / 252
FACTOR_COLS = ['Mkt_RF', 'SMB', 'HML', 'RMW', 'CMA', 'MOM']
EST_START, EST_END = -250, -20
EVENT_WINDOWS = {'m1p1': (-1, 1), 'm3p3': (-3, 3), 'm5p5': (-5, 5),
                  'm10p10': (-10, 10), '0p5': (0, 5)}
MIN_EST_OBS = 150          # minimum non-missing days required within the estimation window
OVERLAP_TD = 20            # trading days: a second filing within this gap is flagged as overlapping
RW_B = 2000                # Romano-Wolf bootstrap replications
CG_CATS = ['AINDEX', 'BINDEX', 'CINDEX', 'DINDEX', 'OINDEX', 'TRINDEX']

print('Event windows:', EVENT_WINDOWS)
print(f'Estimation window: ({EST_START}, {EST_END}) trading days  |  MIN_EST_OBS={MIN_EST_OBS}')

## 1 — Event universe

Every quarterly CG filing for firms in the matched universe, restricted to filings that have
a CG sub-index score and the controls (`Beta_Market`, `Momentum`, `Log_MarketCap`) available
for that firm-quarter. The factor-coverage cutoff (§3) is applied after this initial funnel.

In [ ]:
matched = pd.read_excel(PROC / 'matched_companies.xlsx', usecols=['BSE Code', 'NSE Symbol'])
matched['BSE Code'] = pd.to_numeric(matched['BSE Code'], errors='coerce')

fdb = pd.read_csv(PROC / 'filing_dates_db.csv')
fdb['BSE_Code'] = pd.to_numeric(fdb['BSE_Code'], errors='coerce')
fdb['Filing_Date'] = pd.to_datetime(fdb['Filing_Date'])

events = (fdb.merge(matched, left_on='BSE_Code', right_on='BSE Code', how='inner')
             .drop(columns='BSE_Code'))
print(f'Filings, matched universe          : {len(events):,}  ({events["BSE Code"].nunique()} firms)')

scores = pd.read_csv(PROC / 'cg_scores.csv')
scores['BSE Code'] = pd.to_numeric(scores['BSE Code'], errors='coerce')
wide_scores = scores.pivot_table(index=['BSE Code', 'Q_FY'], columns='Category', values='Avg_Score').reset_index()
wide_scores.columns.name = None

events = events.merge(wide_scores, on=['BSE Code', 'Q_FY'], how='inner')
print(f'+ has a CG score for the filing Q_FY : {len(events):,}')

ctrl = pd.read_csv(PROC / 'controls_quarterly.csv')
ctrl['BSE Code'] = pd.to_numeric(ctrl['BSE Code'], errors='coerce')
events = events.merge(ctrl[['BSE Code', 'Q_FY', 'Beta_Market', 'Momentum', 'Log_MarketCap']],
                       on=['BSE Code', 'Q_FY'], how='inner')
print(f'+ has controls for the filing Q_FY   : {len(events):,}  ({events["BSE Code"].nunique()} firms)')
print(f'Filing date range: {events["Filing_Date"].min().date()} -> {events["Filing_Date"].max().date()}')

## 2 — Overlapping-filing flag

A firm's second filing within 20 trading days of an earlier one would have its estimation
and event windows contaminated by the first filing's own abnormal-return signal. Flagged
here (using calendar-day proximity as a fast proxy, adequate at a 20-trading-day / ~28
calendar-day threshold); **not dropped** from the primary sample — see §7 for the
overlap-excluded robustness spec.

In [ ]:
OVERLAP_CALENDAR_DAYS = 28  # ~20 trading days at a 5-day week, no holiday adjustment

events = events.sort_values(['BSE Code', 'Filing_Date']).reset_index(drop=True)
events['_prev_filing_gap'] = events.groupby('BSE Code')['Filing_Date'].diff().dt.days
events['overlapping'] = events['_prev_filing_gap'] <= OVERLAP_CALENDAR_DAYS

n_overlap = events['overlapping'].sum()
print(f'Overlapping filings (2nd of a pair within ~{OVERLAP_TD} td): {n_overlap:,} / {len(events):,}')
events.drop(columns='_prev_filing_gap', inplace=True)

## 3 — Daily prices and FF5+MOM factors

Daily prices for the full matched universe (bulk download, same approach as
`08_ff5_regression.ipynb`). No daily factor file exists (`ff5mom_factors_daily.csv` is not
present in `data/processed/`), so daily FF5+MOM factors are reconstructed from
`ff5mom_factors_monthly.csv` by spreading each month's return evenly across its trading
days — identical method to `08`.

In [ ]:
imap = pd.read_excel(PROC / 'industry_map.xlsx').dropna(subset=['NSE Symbol'])
tickers_ns = [f'{s}.NS' for s in imap['NSE Symbol']]
print(f'Tickers requested: {len(tickers_ns)}')

px_raw = yf.download(tickers_ns, start='2019-07-01', end='2026-04-01',
                     auto_adjust=True, progress=True)['Close']
valid = px_raw.columns[px_raw.isna().mean() < 0.5].tolist()
px = px_raw[valid]
print(f'Valid tickers: {len(valid)} / {len(tickers_ns)}')

ret_d = np.log(px / px.shift(1)).dropna(how='all')
print(f'Daily returns: {ret_d.shape}  ({ret_d.index.min().date()} -> {ret_d.index.max().date()})')

In [ ]:
factors_m = pd.read_csv(PROC / 'ff5mom_factors_monthly.csv', parse_dates=['Date']).set_index('Date')
print(f'Monthly factor coverage: {factors_m.index.min().date()} -> {factors_m.index.max().date()}')

fac_m = factors_m.copy()
fac_m.index = fac_m.index.year * 100 + fac_m.index.month
daily_key = ret_d.index.year * 100 + ret_d.index.month
td_per_month = pd.Series(1, index=daily_key).groupby(level=0).sum()

fac_for_day = fac_m.reindex(daily_key)
td_for_day = td_per_month.reindex(daily_key).fillna(21).values
factors_d = pd.DataFrame(fac_for_day.values / td_for_day[:, None], index=ret_d.index, columns=FACTOR_COLS)
factors_d['CMA'] = factors_d['CMA'].fillna(0)

trading_days = ret_d.index
factor_covered_days = factors_d.dropna().index
print(f'Trading days with factor coverage: {len(factor_covered_days)} / {len(trading_days)}')

# Earliest filing date whose full (-250,-20 td) estimation window sits inside factor coverage
factor_start = factor_covered_days.min()
start_pos = trading_days.get_indexer([factor_start], method='bfill')[0]
earliest_usable_filing = trading_days[start_pos - EST_START]  # EST_START is negative -> adds |EST_START|
print(f'Factor coverage starts: {factor_start.date()}')
print(f'=> Earliest filing date with a fully-covered estimation window: {earliest_usable_filing.date()}')

before = len(events)
events = events[events['Filing_Date'] >= earliest_usable_filing].reset_index(drop=True)
print(f'\nEvents dropped for insufficient factor coverage: {before - len(events):,}')
print(f'Events remaining: {len(events):,}  ({events["BSE Code"].nunique()} firms)')